In [39]:
import duckdb
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 180)
pd.set_option('display.max_colwidth', None)

con = duckdb.connect(r'data\gold.duckdb', read_only=True)
print('Connected to gold.duckdb')

def q(sql):
    return con.sql(sql).df()

Connected to gold.duckdb


In [40]:
# Last IPL match ingested into gold.duckdb
q(f"""
SELECT m.match_id, m.match_date, m.season, m.team_1, m.team_2,
    coalesce(m.event_stage, 'League')        AS stage,
    coalesce(m.outcome_winner, 'No result')  AS result
FROM main_marts.fct_match_results m
WHERE m.event_name ILIKE '%indian premier league%'
ORDER BY m.match_date DESC LIMIT 1
""")

,match_id,match_date,season,team_1,team_2,stage,result
0,1529286,2026-05-01,2026,Rajasthan Royals,Delhi Capitals,League,Delhi Capitals


In [41]:
# ── Set team and season here ─────────────────────────────────────────────────
TEAM   = "Sunrisers Hyderabad"
SEASON = '2026'

# For teams that changed official name add alternate names here.
# Example for RCB: TEAM_ALIASES = ["Royal Challengers Bengaluru"]
TEAM_ALIASES = []
# ─────────────────────────────────────────────────────────────────────────────

_names  = [TEAM] + TEAM_ALIASES
IN_LIST = ", ".join(f"'{t}'" for t in _names)
print(f"Team: {TEAM}  |  Season: {SEASON}")

Team: Sunrisers Hyderabad  |  Season: 2026


## Season Record

In [42]:
# Match-by-match results for the season
q(f"""
SELECT
    m.match_date,
    CASE WHEN m.team_1 IN ({IN_LIST}) THEN m.team_2 ELSE m.team_1 END  AS vs,
    coalesce(m.event_stage, 'League')                                   AS stage,
    CASE WHEN m.outcome_winner IN ({IN_LIST}) THEN 'W'
         WHEN m.is_no_result THEN 'NR'
         WHEN m.is_tie       THEN 'T'
         ELSE 'L' END                                                   AS result,
    m.outcome_by_runs                                                   AS by_runs,
    m.outcome_by_wickets                                                AS by_wkts,
    m.outcome_method
FROM main_marts.fct_match_results m
WHERE (m.team_1 IN ({IN_LIST}) OR m.team_2 IN ({IN_LIST}))
  AND m.event_name ILIKE '%indian premier league%'
  AND m.season = '{SEASON}'
ORDER BY m.match_date
""")

,match_date,vs,stage,result,by_runs,by_wkts,outcome_method
0,2026-03-28,Royal Challengers Bengaluru,League,L,<NA>,6,None
1,2026-04-02,Kolkata Knight Riders,League,W,65,<NA>,None
2,2026-04-05,Lucknow Super Giants,League,L,<NA>,5,None
3,2026-04-11,Punjab Kings,League,L,<NA>,6,None
4,2026-04-13,Rajasthan Royals,League,W,57,<NA>,None
5,2026-04-18,Chennai Super Kings,League,W,10,<NA>,None
6,2026-04-21,Delhi Capitals,League,W,47,<NA>,None
7,2026-04-25,Rajasthan Royals,League,W,<NA>,5,None
8,2026-04-29,Mumbai Indians,League,W,<NA>,6,None


In [43]:
# Season summary — W/L/NR
q(f"""
SELECT
    count(*)                                                                    AS played,
    sum(CASE WHEN m.outcome_winner IN ({IN_LIST}) THEN 1 ELSE 0 END)            AS won,
    sum(CASE WHEN m.has_result AND m.outcome_winner NOT IN ({IN_LIST})
         THEN 1 ELSE 0 END)                                                     AS lost,
    sum(m.is_tie::integer)                                                      AS tied,
    sum(m.is_no_result::integer)                                                AS nr,
    round(sum(CASE WHEN m.outcome_winner IN ({IN_LIST}) THEN 1 ELSE 0 END) * 100.0
          / nullif(sum(m.has_result::integer), 0), 1)                           AS win_pct
FROM main_marts.fct_match_results m
WHERE (m.team_1 IN ({IN_LIST}) OR m.team_2 IN ({IN_LIST}))
  AND m.event_name ILIKE '%indian premier league%'
  AND m.season = '{SEASON}'
""")

,played,won,lost,tied,nr,win_pct
0,9,6.0,3.0,NaN,NaN,66.7


## All Teams — Top 3 Batting Concentration

In [44]:
# All teams — % of runs scored by top 3 batters, ordered by concentration
q(f"""
WITH batter_runs AS (
    SELECT
        b.batting_team                                      AS team,
        b.player_name                                       AS batter,
        sum(b.runs)                                         AS runs
    FROM main_intermediate.int_batting_by_innings b
    JOIN main_marts.fct_match_results m ON b.match_id = m.match_id
    WHERE m.event_name ILIKE '%indian premier league%'
      AND m.season = '{SEASON}'
      AND b.innings_number IN (1, 2)
      AND NOT b.super_over
    GROUP BY b.batting_team, b.player_name
),
ranked AS (
    SELECT team, batter, runs,
        row_number() OVER (PARTITION BY team ORDER BY runs DESC) AS rnk
    FROM batter_runs
),
team_totals AS (
    SELECT team, sum(runs) AS total_runs
    FROM batter_runs
    GROUP BY team
),
top3 AS (
    SELECT team,
        sum(runs)                                                                       AS top3_runs,
        string_agg(batter || ' (' || runs::integer || ')', ', ' ORDER BY runs DESC)    AS top3_players
    FROM ranked
    WHERE rnk <= 3
    GROUP BY team
)
SELECT
    t.team,
    t.total_runs,
    x.top3_runs,
    round(x.top3_runs * 100.0 / t.total_runs, 1)   AS top3_pct,
    x.top3_players
FROM team_totals t
JOIN top3 x ON t.team = x.team
ORDER BY top3_pct DESC
""")

,team,total_runs,top3_runs,top3_pct,top3_players
0,Gujarat Titans,1489.0,1010.0,67.8,"Shubman Gill (373), B Sai Sudharsan (328), JC Buttler (309)"
1,Punjab Kings,1477.0,938.0,63.5,"P Simran Singh (346), SS Iyer (309), Priyansh Arya (283)"
2,Sunrisers Hyderabad,1835.0,1151.0,62.7,"Abhishek Sharma (425), H Klaasen (414), Ishan Kishan (312)"
3,Royal Challengers Bengaluru,1578.0,918.0,58.2,"V Kohli (379), D Padikkal (282), RM Patidar (257)"
4,Rajasthan Royals,1765.0,1006.0,57.0,"V Suryavanshi (404), YBK Jaiswal (312), Dhruv Jurel (290)"
5,Delhi Capitals,1560.0,861.0,55.2,"KL Rahul (433), T Stubbs (219), P Nissanka (209)"
6,Kolkata Knight Riders,1176.0,620.0,52.7,"RK Singh (215), A Raghuvanshi (209), C Green (196)"
7,Lucknow Super Giants,1201.0,594.0,49.5,"MR Marsh (212), AK Markram (193), RR Pant (189)"
8,Chennai Super Kings,1399.0,683.0,48.8,"SV Samson (304), A Mhatre (201), RD Gaikwad (178)"
9,Mumbai Indians,1408.0,624.0,44.3,"RD Rickelton (260), Tilak Varma (188), Naman Dhir (176)"


In [ ]:
# Individual batting dominance — players carrying ≥ PCT_THRESHOLD% of team runs
# (33.33 means one batter scoring more than 1/3 of all team runs)
PCT_THRESHOLD = 25.0   # lower if no results; no player hit 33.33 in IPL 2026

q(f"""
WITH batter_runs AS (
    SELECT
        b.batting_team  AS team,
        b.player_name   AS batter,
        sum(b.runs)     AS runs
    FROM main_intermediate.int_batting_by_innings b
    JOIN main_marts.fct_match_results m ON b.match_id = m.match_id
    WHERE m.event_name ILIKE '%indian premier league%'
      AND m.season = '{SEASON}'
      AND b.innings_number IN (1, 2)
      AND NOT b.super_over
    GROUP BY b.batting_team, b.player_name
),
team_totals AS (
    SELECT team, sum(runs) AS total_runs
    FROM batter_runs
    GROUP BY team
)
SELECT
    br.team,
    br.batter,
    br.runs,
    t.total_runs,
    round(br.runs * 100.0 / t.total_runs, 1)  AS pct_of_team_runs
FROM batter_runs br
JOIN team_totals t ON br.team = t.team
WHERE br.runs * 100.0 / t.total_runs >= {PCT_THRESHOLD}
ORDER BY pct_of_team_runs DESC
""")

In [ ]:
# All teams — % of wickets taken by top 3 bowlers, ordered by concentration
q(f"""
WITH bowler_wkts AS (
    SELECT
        b.bowling_team              AS team,
        b.player_name               AS bowler,
        sum(b.wickets_credited)     AS wkts
    FROM main_intermediate.int_bowling_by_innings b
    JOIN main_marts.fct_match_results m ON b.match_id = m.match_id
    WHERE m.event_name ILIKE '%indian premier league%'
      AND m.season = '{SEASON}'
      AND b.innings_number IN (1, 2)
      AND NOT b.super_over
    GROUP BY b.bowling_team, b.player_name
),
ranked AS (
    SELECT team, bowler, wkts,
        row_number() OVER (PARTITION BY team ORDER BY wkts DESC) AS rnk
    FROM bowler_wkts
),
team_totals AS (
    SELECT team, sum(wkts) AS total_wkts
    FROM bowler_wkts
    GROUP BY team
),
top3 AS (
    SELECT team,
        sum(wkts)                                                                       AS top3_wkts,
        string_agg(bowler || ' (' || wkts::integer || ')', ', ' ORDER BY wkts DESC)    AS top3_bowlers
    FROM ranked
    WHERE rnk <= 3
    GROUP BY team
)
SELECT
    t.team,
    t.total_wkts,
    x.top3_wkts,
    round(x.top3_wkts * 100.0 / t.total_wkts, 1)  AS top3_pct,
    x.top3_bowlers
FROM team_totals t
JOIN top3 x ON t.team = x.team
ORDER BY top3_pct DESC
""")

In [ ]:
# All teams — batting profile (ordered by avg score)
q(f"""
SELECT
    d.batting_team                                                                      AS team,
    count(DISTINCT d.match_id)                                                          AS mat,
    round(sum(d.runs_batter)::double / count(DISTINCT d.match_id), 1)                  AS avg_score,
    round(sum(d.runs_batter) * 100.0
          / nullif(sum(CASE WHEN d.extras_wides = 0 THEN 1 ELSE 0 END), 0), 1)         AS avg_sr,
    round(sum(CASE WHEN d.extras_wides = 0 THEN 1 ELSE 0 END)::double
          / nullif(sum(CASE WHEN d.runs_batter = 4 AND NOT d.runs_non_boundary THEN 1 ELSE 0 END), 0), 1)
                                                                                        AS balls_per_4,
    round(sum(CASE WHEN d.extras_wides = 0 THEN 1 ELSE 0 END)::double
          / nullif(sum(CASE WHEN d.runs_batter = 6 AND NOT d.runs_non_boundary THEN 1 ELSE 0 END), 0), 1)
                                                                                        AS balls_per_6,
    round(sum(CASE WHEN d.runs_batter = 0 AND d.extras_wides = 0 THEN 1 ELSE 0 END) * 100.0
          / nullif(sum(CASE WHEN d.extras_wides = 0 THEN 1 ELSE 0 END), 0), 1)         AS dot_pct,
    round(sum(CASE WHEN d.runs_batter IN (4, 6) AND NOT d.runs_non_boundary THEN 1 ELSE 0 END) * 100.0
          / nullif(sum(CASE WHEN d.extras_wides = 0 THEN 1 ELSE 0 END), 0), 1)         AS boundary_pct
FROM main_marts.fct_deliveries d
JOIN main_marts.fct_match_results m ON d.match_id = m.match_id
WHERE m.event_name ILIKE '%indian premier league%'
  AND m.season = '{SEASON}'
  AND d.innings_number IN (1, 2)
GROUP BY d.batting_team
ORDER BY avg_score DESC
""")

In [ ]:
# All teams — batting by phase, pivoted (PP / Middle / Death)
q(f"""
SELECT
    d.batting_team                                                                      AS team,
    round(sum(CASE WHEN d.over_number BETWEEN 0  AND 5
              THEN d.runs_total ELSE 0 END)::double / count(DISTINCT d.match_id), 1)    AS pp_runs,
    round(sum(CASE WHEN d.over_number BETWEEN 0  AND 5
              THEN d.wicket_count ELSE 0 END)::double / count(DISTINCT d.match_id), 2)  AS pp_wkts,
    round(sum(CASE WHEN d.over_number BETWEEN 0  AND 5
              THEN d.runs_total ELSE 0 END) * 6.0
          / nullif(sum(CASE WHEN d.over_number BETWEEN 0 AND 5
                             AND d.extras_wides=0 AND d.extras_noballs=0
                        THEN 1 ELSE 0 END), 0), 2)                                     AS pp_rr,
    round(sum(CASE WHEN d.over_number BETWEEN 0 AND 5
                    AND d.runs_batter IN (4, 6) AND NOT d.runs_non_boundary
              THEN 1 ELSE 0 END) * 100.0
          / nullif(sum(CASE WHEN d.over_number BETWEEN 0 AND 5
                             AND d.extras_wides=0 THEN 1 ELSE 0 END), 0), 1)           AS pp_boundary_pct,
    round(sum(CASE WHEN d.over_number BETWEEN 6  AND 14
              THEN d.runs_total ELSE 0 END)::double / count(DISTINCT d.match_id), 1)    AS mid_runs,
    round(sum(CASE WHEN d.over_number BETWEEN 6  AND 14
              THEN d.wicket_count ELSE 0 END)::double / count(DISTINCT d.match_id), 2)  AS mid_wkts,
    round(sum(CASE WHEN d.over_number BETWEEN 6  AND 14
              THEN d.runs_total ELSE 0 END) * 6.0
          / nullif(sum(CASE WHEN d.over_number BETWEEN 6 AND 14
                             AND d.extras_wides=0 AND d.extras_noballs=0
                        THEN 1 ELSE 0 END), 0), 2)                                     AS mid_rr,
    round(sum(CASE WHEN d.over_number BETWEEN 6 AND 14
                    AND d.runs_batter IN (4, 6) AND NOT d.runs_non_boundary
              THEN 1 ELSE 0 END) * 100.0
          / nullif(sum(CASE WHEN d.over_number BETWEEN 6 AND 14
                             AND d.extras_wides=0 THEN 1 ELSE 0 END), 0), 1)           AS mid_boundary_pct,
    round(sum(CASE WHEN d.over_number BETWEEN 15 AND 19
              THEN d.runs_total ELSE 0 END)::double / count(DISTINCT d.match_id), 1)    AS death_runs,
    round(sum(CASE WHEN d.over_number BETWEEN 15 AND 19
              THEN d.wicket_count ELSE 0 END)::double / count(DISTINCT d.match_id), 2)  AS death_wkts,
    round(sum(CASE WHEN d.over_number BETWEEN 15 AND 19
              THEN d.runs_total ELSE 0 END) * 6.0
          / nullif(sum(CASE WHEN d.over_number BETWEEN 15 AND 19
                             AND d.extras_wides=0 AND d.extras_noballs=0
                        THEN 1 ELSE 0 END), 0), 2)                                     AS death_rr,
    round(sum(CASE WHEN d.over_number BETWEEN 15 AND 19
                    AND d.runs_batter IN (4, 6) AND NOT d.runs_non_boundary
              THEN 1 ELSE 0 END) * 100.0
          / nullif(sum(CASE WHEN d.over_number BETWEEN 15 AND 19
                             AND d.extras_wides=0 THEN 1 ELSE 0 END), 0), 1)           AS death_boundary_pct
FROM main_marts.fct_deliveries d
JOIN main_marts.fct_match_results m ON d.match_id = m.match_id
WHERE m.event_name ILIKE '%indian premier league%'
  AND m.season = '{SEASON}'
  AND d.innings_number IN (1, 2)
  AND d.over_number BETWEEN 0 AND 19
GROUP BY d.batting_team
ORDER BY pp_runs DESC
""")

In [ ]:
# All teams — bowling profile (ordered by economy)
q(f"""
SELECT
    d.bowling_team                                                                      AS team,
    round(sum(CASE WHEN d.extras_wides=0 AND d.extras_noballs=0 THEN 1 ELSE 0 END)::double
          / nullif(sum(d.wicket_count), 0), 1)                                          AS balls_per_wkt,
    round(sum(d.runs_total) * 6.0
          / nullif(sum(CASE WHEN d.extras_wides=0 AND d.extras_noballs=0 THEN 1 ELSE 0 END), 0), 2)
                                                                                        AS economy,
    round(sum(CASE WHEN d.runs_batter=0 AND d.extras_wides=0 THEN 1 ELSE 0 END) * 100.0
          / nullif(sum(CASE WHEN d.extras_wides=0 THEN 1 ELSE 0 END), 0), 1)           AS dot_pct,
    round((sum(d.extras_wides) + sum(d.extras_noballs))::double
          / count(DISTINCT d.match_id), 2)                                              AS extras_per_match
FROM main_marts.fct_deliveries d
JOIN main_marts.fct_match_results m ON d.match_id = m.match_id
WHERE m.event_name ILIKE '%indian premier league%'
  AND m.season = '{SEASON}'
  AND d.innings_number IN (1, 2)
GROUP BY d.bowling_team
ORDER BY economy ASC
""")

In [ ]:
# All teams — bowling by phase, pivoted (PP / Middle / Death)
q(f"""
SELECT
    d.bowling_team                                                                      AS team,
    round(sum(CASE WHEN d.over_number BETWEEN 0  AND 5
              THEN d.runs_total ELSE 0 END)::double / count(DISTINCT d.match_id), 1)    AS pp_conceded,
    round(sum(CASE WHEN d.over_number BETWEEN 0  AND 5
              THEN d.wicket_count ELSE 0 END)::double / count(DISTINCT d.match_id), 2)  AS pp_wkts,
    round(sum(CASE WHEN d.over_number BETWEEN 0  AND 5
              THEN d.runs_total ELSE 0 END) * 6.0
          / nullif(sum(CASE WHEN d.over_number BETWEEN 0 AND 5
                             AND d.extras_wides=0 AND d.extras_noballs=0
                        THEN 1 ELSE 0 END), 0), 2)                                     AS pp_econ,
    round(sum(CASE WHEN d.over_number BETWEEN 0 AND 5
                    AND d.runs_batter IN (4, 6) AND NOT d.runs_non_boundary
              THEN 1 ELSE 0 END) * 100.0
          / nullif(sum(CASE WHEN d.over_number BETWEEN 0 AND 5
                             AND d.extras_wides=0 THEN 1 ELSE 0 END), 0), 1)           AS pp_boundary_pct,
    round(sum(CASE WHEN d.over_number BETWEEN 6  AND 14
              THEN d.runs_total ELSE 0 END)::double / count(DISTINCT d.match_id), 1)    AS mid_conceded,
    round(sum(CASE WHEN d.over_number BETWEEN 6  AND 14
              THEN d.wicket_count ELSE 0 END)::double / count(DISTINCT d.match_id), 2)  AS mid_wkts,
    round(sum(CASE WHEN d.over_number BETWEEN 6  AND 14
              THEN d.runs_total ELSE 0 END) * 6.0
          / nullif(sum(CASE WHEN d.over_number BETWEEN 6 AND 14
                             AND d.extras_wides=0 AND d.extras_noballs=0
                        THEN 1 ELSE 0 END), 0), 2)                                     AS mid_econ,
    round(sum(CASE WHEN d.over_number BETWEEN 6 AND 14
                    AND d.runs_batter IN (4, 6) AND NOT d.runs_non_boundary
              THEN 1 ELSE 0 END) * 100.0
          / nullif(sum(CASE WHEN d.over_number BETWEEN 6 AND 14
                             AND d.extras_wides=0 THEN 1 ELSE 0 END), 0), 1)           AS mid_boundary_pct,
    round(sum(CASE WHEN d.over_number BETWEEN 15 AND 19
              THEN d.runs_total ELSE 0 END)::double / count(DISTINCT d.match_id), 1)    AS death_conceded,
    round(sum(CASE WHEN d.over_number BETWEEN 15 AND 19
              THEN d.wicket_count ELSE 0 END)::double / count(DISTINCT d.match_id), 2)  AS death_wkts,
    round(sum(CASE WHEN d.over_number BETWEEN 15 AND 19
              THEN d.runs_total ELSE 0 END) * 6.0
          / nullif(sum(CASE WHEN d.over_number BETWEEN 15 AND 19
                             AND d.extras_wides=0 AND d.extras_noballs=0
                        THEN 1 ELSE 0 END), 0), 2)                                     AS death_econ,
    round(sum(CASE WHEN d.over_number BETWEEN 15 AND 19
                    AND d.runs_batter IN (4, 6) AND NOT d.runs_non_boundary
              THEN 1 ELSE 0 END) * 100.0
          / nullif(sum(CASE WHEN d.over_number BETWEEN 15 AND 19
                             AND d.extras_wides=0 THEN 1 ELSE 0 END), 0), 1)           AS death_boundary_pct
FROM main_marts.fct_deliveries d
JOIN main_marts.fct_match_results m ON d.match_id = m.match_id
WHERE m.event_name ILIKE '%indian premier league%'
  AND m.season = '{SEASON}'
  AND d.innings_number IN (1, 2)
  AND d.over_number BETWEEN 0 AND 19
GROUP BY d.bowling_team
ORDER BY pp_econ ASC
""")

In [ ]:
# All teams — batting & bowling phase averages split by match result
q(f"""
WITH innings_scores AS (
    SELECT
        d.batting_team,
        d.bowling_team,
        d.match_id,
        d.over_number,
        d.runs_total,
        d.wicket_count,
        m.outcome_winner
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m ON d.match_id = m.match_id
    WHERE m.event_name ILIKE '%indian premier league%'
      AND m.season = '{SEASON}'
      AND d.innings_number IN (1, 2)
      AND d.over_number BETWEEN 0 AND 19
      AND m.has_result
),
bat_agg AS (
    SELECT batting_team AS team, match_id,
        outcome_winner = batting_team                                            AS won,
        sum(runs_total)                                                          AS score,
        sum(CASE WHEN over_number BETWEEN 0  AND 5  THEN runs_total ELSE 0 END) AS pp_score,
        sum(CASE WHEN over_number BETWEEN 6  AND 14 THEN runs_total ELSE 0 END) AS mid_score,
        sum(CASE WHEN over_number BETWEEN 15 AND 19 THEN runs_total ELSE 0 END) AS death_score
    FROM innings_scores GROUP BY batting_team, match_id, outcome_winner
),
bowl_agg AS (
    SELECT bowling_team AS team, match_id,
        outcome_winner != bowling_team                                           AS won,
        sum(runs_total)                                                          AS conceded,
        sum(CASE WHEN over_number BETWEEN 0  AND 5  THEN runs_total ELSE 0 END) AS pp_conc,
        sum(CASE WHEN over_number BETWEEN 6  AND 14 THEN runs_total ELSE 0 END) AS mid_conc,
        sum(CASE WHEN over_number BETWEEN 15 AND 19 THEN runs_total ELSE 0 END) AS death_conc
    FROM innings_scores GROUP BY bowling_team, match_id, outcome_winner
)
SELECT
    b.team,
    sum(b.won::integer)                                                         AS wins,
    sum((NOT b.won)::integer)                                                   AS losses,
    round(avg(CASE WHEN b.won  THEN b.score       END), 1)                     AS bat_avg_W,
    round(avg(CASE WHEN NOT b.won THEN b.score    END), 1)                     AS bat_avg_L,
    round(avg(CASE WHEN b.won  THEN b.pp_score    END), 1)                     AS bat_pp_W,
    round(avg(CASE WHEN NOT b.won THEN b.pp_score END), 1)                     AS bat_pp_L,
    round(avg(CASE WHEN bw.won THEN bw.conceded   END), 1)                     AS bowl_avg_W,
    round(avg(CASE WHEN NOT bw.won THEN bw.conceded END), 1)                   AS bowl_avg_L,
    round(avg(CASE WHEN bw.won THEN bw.pp_conc    END), 1)                     AS bowl_pp_W,
    round(avg(CASE WHEN NOT bw.won THEN bw.pp_conc END), 1)                    AS bowl_pp_L
FROM bat_agg b
JOIN bowl_agg bw ON b.team = bw.team AND b.match_id = bw.match_id
GROUP BY b.team
ORDER BY wins DESC
""")

## Batting Contributions

In [45]:
# All batters — season batting summary (super overs excluded)
q(f"""
SELECT
    b.player_name                                                               AS batter,
    count(DISTINCT b.match_id)                                                  AS mat,
    count(*)                                                                    AS inns,
    sum(b.not_out::integer)                                                     AS no,
    sum(b.runs)                                                                 AS runs,
    max(b.runs)                                                                 AS hs,
    round(sum(b.runs)::double / nullif(sum(b.dismissals), 0), 2)                AS avg,
    sum(b.balls_faced)                                                          AS bf,
    round(sum(b.runs) * 100.0 / nullif(sum(b.balls_faced), 0), 2)               AS sr,
    sum(CASE WHEN b.runs >= 50 AND b.runs < 100 THEN 1 ELSE 0 END)              AS "50s",
    sum(CASE WHEN b.runs >= 100 THEN 1 ELSE 0 END)                              AS "100s",
    sum(b.fours)                                                                AS "4s",
    sum(b.sixes)                                                                AS "6s"
FROM main_intermediate.int_batting_by_innings b
JOIN main_marts.fct_match_results m ON b.match_id = m.match_id
WHERE b.batting_team IN ({IN_LIST})
  AND m.event_name ILIKE '%indian premier league%'
  AND m.season = '{SEASON}'
  AND b.innings_number IN (1, 2)
  AND NOT b.super_over
GROUP BY b.player_name
ORDER BY runs DESC
""")

,batter,mat,inns,no,runs,hs,avg,bf,sr,50s,100s,4s,6s
0,Abhishek Sharma,9,9,1.0,425.0,135.0,53.13,229.0,185.59,3.0,1.0,40.0,31.0
1,H Klaasen,9,9,2.0,414.0,65.0,59.14,279.0,148.39,4.0,0.0,32.0,18.0
2,Ishan Kishan,9,9,0.0,312.0,91.0,34.67,163.0,191.41,3.0,0.0,35.0,16.0
3,TM Head,9,9,0.0,262.0,76.0,29.11,166.0,157.83,1.0,0.0,26.0,16.0
4,Nithish Kumar Reddy,8,8,1.0,193.0,56.0,27.57,122.0,158.20,1.0,0.0,12.0,14.0
5,S Arora,7,7,3.0,93.0,30.0,23.25,55.0,169.09,0.0,0.0,6.0,8.0
6,Aniket Verma,7,7,2.0,73.0,43.0,14.60,49.0,148.98,0.0,0.0,5.0,5.0
7,Shivang Kumar,3,3,0.0,21.0,12.0,7.00,15.0,140.00,0.0,0.0,3.0,1.0
8,LS Livingstone,2,2,0.0,15.0,14.0,7.50,26.0,57.69,0.0,0.0,0.0,1.0
9,Harsh Dubey,5,5,3.0,13.0,9.0,6.50,11.0,118.18,0.0,0.0,2.0,0.0


In [46]:
# Top 3 batters — share of team runs this season
q(f"""
WITH team_runs AS (
    SELECT sum(b.runs) AS total_runs
    FROM main_intermediate.int_batting_by_innings b
    JOIN main_marts.fct_match_results m ON b.match_id = m.match_id
    WHERE b.batting_team IN ({IN_LIST})
      AND m.event_name ILIKE '%indian premier league%'
      AND m.season = '{SEASON}'
      AND b.innings_number IN (1, 2)
      AND NOT b.super_over
)
SELECT
    b.player_name                                                              AS batter,
    sum(b.runs)                                                                AS runs,
    round(sum(b.runs) * 100.0 / (SELECT total_runs FROM team_runs), 1)        AS pct_of_team_runs
FROM main_intermediate.int_batting_by_innings b
JOIN main_marts.fct_match_results m ON b.match_id = m.match_id
WHERE b.batting_team IN ({IN_LIST})
  AND m.event_name ILIKE '%indian premier league%'
  AND m.season = '{SEASON}'
  AND b.innings_number IN (1, 2)
  AND NOT b.super_over
GROUP BY b.player_name
ORDER BY runs DESC
LIMIT 3
""")

,batter,runs,pct_of_team_runs
0,Abhishek Sharma,425.0,23.2
1,H Klaasen,414.0,22.6
2,Ishan Kishan,312.0,17.0


In [ ]:
# Team batting profile — overall season metrics
q(f"""
SELECT
    count(DISTINCT d.match_id)                                                          AS matches,
    sum(d.runs_batter)                                                                  AS total_runs,
    round(sum(d.runs_batter)::double / count(DISTINCT d.match_id), 1)                  AS avg_score,
    round(sum(d.runs_batter) * 100.0
          / nullif(sum(CASE WHEN d.extras_wides = 0 THEN 1 ELSE 0 END), 0), 1)         AS avg_sr,
    round(sum(CASE WHEN d.extras_wides = 0 THEN 1 ELSE 0 END)::double
          / nullif(sum(CASE WHEN d.runs_batter = 4 AND NOT d.runs_non_boundary THEN 1 ELSE 0 END), 0), 1)
                                                                                        AS balls_per_four,
    round(sum(CASE WHEN d.extras_wides = 0 THEN 1 ELSE 0 END)::double
          / nullif(sum(CASE WHEN d.runs_batter = 6 AND NOT d.runs_non_boundary THEN 1 ELSE 0 END), 0), 1)
                                                                                        AS balls_per_six,
    round(sum(CASE WHEN d.runs_batter = 0 AND d.extras_wides = 0 THEN 1 ELSE 0 END) * 100.0
          / nullif(sum(CASE WHEN d.extras_wides = 0 THEN 1 ELSE 0 END), 0), 1)         AS dot_pct,
    round(sum(CASE WHEN d.runs_batter IN (4, 6) AND NOT d.runs_non_boundary THEN 1 ELSE 0 END) * 100.0
          / nullif(sum(CASE WHEN d.extras_wides = 0 THEN 1 ELSE 0 END), 0), 1)         AS boundary_pct
FROM main_marts.fct_deliveries d
JOIN main_marts.fct_match_results m ON d.match_id = m.match_id
WHERE d.batting_team IN ({IN_LIST})
  AND m.event_name ILIKE '%indian premier league%'
  AND m.season = '{SEASON}'
  AND d.innings_number IN (1, 2)
""")

In [ ]:
# Batting by phase — Powerplay / Middle / Death
q(f"""
SELECT
    CASE WHEN d.over_number BETWEEN 0 AND 5   THEN '1. Powerplay (1-6)'
         WHEN d.over_number BETWEEN 6 AND 14  THEN '2. Middle (7-15)'
         WHEN d.over_number BETWEEN 15 AND 19 THEN '3. Death (16-20)'
    END                                                                                  AS phase,
    round(sum(d.runs_total)::double          / count(DISTINCT d.match_id), 1)            AS avg_runs,
    round(sum(d.wicket_count)::double        / count(DISTINCT d.match_id), 2)            AS avg_wkts_lost,
    round(sum(d.runs_total) * 6.0
          / nullif(sum(CASE WHEN d.extras_wides=0 AND d.extras_noballs=0 THEN 1 ELSE 0 END), 0), 2)
                                                                                         AS run_rate,
    round(sum(CASE WHEN d.runs_batter = 0 AND d.extras_wides = 0 THEN 1 ELSE 0 END) * 100.0
          / nullif(sum(CASE WHEN d.extras_wides = 0 THEN 1 ELSE 0 END), 0), 1)          AS dot_pct,
    round(sum(CASE WHEN d.runs_batter IN (4, 6) AND NOT d.runs_non_boundary THEN 1 ELSE 0 END) * 100.0
          / nullif(sum(CASE WHEN d.extras_wides = 0 THEN 1 ELSE 0 END), 0), 1)          AS boundary_pct
FROM main_marts.fct_deliveries d
JOIN main_marts.fct_match_results m ON d.match_id = m.match_id
WHERE d.batting_team IN ({IN_LIST})
  AND m.event_name ILIKE '%indian premier league%'
  AND m.season = '{SEASON}'
  AND d.innings_number IN (1, 2)
  AND d.over_number BETWEEN 0 AND 19
GROUP BY phase
ORDER BY phase
""")

## Bowling Contributions

In [47]:
# All bowlers — season bowling summary (super overs excluded)
q(f"""
SELECT
    b.player_name                                                               AS bowler,
    count(DISTINCT b.match_id)                                                  AS mat,
    count(*)                                                                    AS inns,
    (floor(sum(b.legal_deliveries) / 6)::integer
     || '.' || (sum(b.legal_deliveries) % 6))::varchar                         AS overs,
    sum(b.legal_deliveries)                                                     AS balls,
    sum(b.runs_conceded)                                                        AS runs,
    sum(b.wickets_credited)                                                     AS wkts,
    round(sum(b.runs_conceded)::double / nullif(sum(b.wickets_credited), 0), 2) AS avg,
    round(sum(b.runs_conceded) * 6.0 / nullif(sum(b.legal_deliveries), 0), 2)  AS econ,
    round(sum(b.legal_deliveries)::double
          / nullif(sum(b.wickets_credited), 0), 2)                              AS sr,
    sum(CASE WHEN b.wickets_credited >= 4 THEN 1 ELSE 0 END)                   AS "4w+"
FROM main_intermediate.int_bowling_by_innings b
JOIN main_marts.fct_match_results m ON b.match_id = m.match_id
WHERE b.bowling_team IN ({IN_LIST})
  AND m.event_name ILIKE '%indian premier league%'
  AND m.season = '{SEASON}'
  AND b.innings_number IN (1, 2)
  AND NOT b.super_over
GROUP BY b.player_name
ORDER BY wkts DESC
""")

,bowler,mat,inns,overs,balls,runs,wkts,avg,econ,sr,4w+
0,E Malinga,9,9,31.0,186.0,287.0,15.0,19.13,9.26,12.40,1.0
1,PP Hinge,4,4,16.0,96.0,199.0,8.0,24.88,12.44,12.00,1.0
2,Harsh Dubey,7,7,20.0,120.0,194.0,8.0,24.25,9.70,15.00,0.0
3,Sakib Hussain,5,5,19.0,114.0,186.0,8.0,23.25,9.79,14.25,1.0
4,Nithish Kumar Reddy,9,9,21.0,126.0,230.0,6.0,38.33,10.95,21.00,0.0
5,Shivang Kumar,7,7,22.0,132.0,210.0,5.0,42.00,9.55,26.40,0.0
6,JD Unadkat,4,4,12.5,77.0,145.0,4.0,36.25,11.30,19.25,0.0
7,DA Payne,2,2,5.0,30.0,70.0,2.0,35.00,14.00,15.00,0.0
8,PJ Cummins,2,2,8.0,48.0,66.0,1.0,66.00,8.25,48.00,0.0
9,D Madushanka,1,1,4.0,24.0,36.0,1.0,36.00,9.00,24.00,0.0


In [48]:
# Top 3 bowlers — share of team wickets this season
q(f"""
WITH team_wkts AS (
    SELECT sum(b.wickets_credited) AS total_wkts
    FROM main_intermediate.int_bowling_by_innings b
    JOIN main_marts.fct_match_results m ON b.match_id = m.match_id
    WHERE b.bowling_team IN ({IN_LIST})
      AND m.event_name ILIKE '%indian premier league%'
      AND m.season = '{SEASON}'
      AND b.innings_number IN (1, 2)
      AND NOT b.super_over
)
SELECT
    b.player_name                                                                   AS bowler,
    sum(b.wickets_credited)                                                         AS wkts,
    round(sum(b.wickets_credited) * 100.0 / (SELECT total_wkts FROM team_wkts), 1) AS pct_of_team_wkts
FROM main_intermediate.int_bowling_by_innings b
JOIN main_marts.fct_match_results m ON b.match_id = m.match_id
WHERE b.bowling_team IN ({IN_LIST})
  AND m.event_name ILIKE '%indian premier league%'
  AND m.season = '{SEASON}'
  AND b.innings_number IN (1, 2)
  AND NOT b.super_over
GROUP BY b.player_name
ORDER BY wkts DESC
LIMIT 3
""")

,bowler,wkts,pct_of_team_wkts
0,E Malinga,15.0,25.9
1,PP Hinge,8.0,13.8
2,Harsh Dubey,8.0,13.8


In [ ]:
# Team bowling profile — overall season metrics
q(f"""
SELECT
    count(DISTINCT d.match_id)                                                          AS matches,
    sum(d.wicket_count)                                                                 AS total_wkts,
    round(sum(CASE WHEN d.extras_wides=0 AND d.extras_noballs=0 THEN 1 ELSE 0 END)::double
          / nullif(sum(d.wicket_count), 0), 1)                                          AS balls_per_wkt,
    round(sum(d.runs_total) * 6.0
          / nullif(sum(CASE WHEN d.extras_wides=0 AND d.extras_noballs=0 THEN 1 ELSE 0 END), 0), 2)
                                                                                        AS avg_economy,
    round(sum(CASE WHEN d.runs_batter = 0 AND d.extras_wides = 0 THEN 1 ELSE 0 END) * 100.0
          / nullif(sum(CASE WHEN d.extras_wides = 0 THEN 1 ELSE 0 END), 0), 1)         AS dot_pct,
    round((sum(d.extras_wides) + sum(d.extras_noballs))::double
          / nullif(count(DISTINCT d.match_id), 0), 2)                                  AS extras_per_match
FROM main_marts.fct_deliveries d
JOIN main_marts.fct_match_results m ON d.match_id = m.match_id
WHERE d.bowling_team IN ({IN_LIST})
  AND m.event_name ILIKE '%indian premier league%'
  AND m.season = '{SEASON}'
  AND d.innings_number IN (1, 2)
""")

In [ ]:
# Bowling by phase — Powerplay / Middle / Death
q(f"""
SELECT
    CASE WHEN d.over_number BETWEEN 0 AND 5   THEN '1. Powerplay (1-6)'
         WHEN d.over_number BETWEEN 6 AND 14  THEN '2. Middle (7-15)'
         WHEN d.over_number BETWEEN 15 AND 19 THEN '3. Death (16-20)'
    END                                                                                  AS phase,
    round(sum(d.runs_total)::double          / count(DISTINCT d.match_id), 1)            AS avg_runs_conceded,
    round(sum(d.wicket_count)::double        / count(DISTINCT d.match_id), 2)            AS avg_wkts_taken,
    round(sum(d.runs_total) * 6.0
          / nullif(sum(CASE WHEN d.extras_wides=0 AND d.extras_noballs=0 THEN 1 ELSE 0 END), 0), 2)
                                                                                         AS economy,
    round(sum(CASE WHEN d.runs_batter = 0 AND d.extras_wides = 0 THEN 1 ELSE 0 END) * 100.0
          / nullif(sum(CASE WHEN d.extras_wides = 0 THEN 1 ELSE 0 END), 0), 1)          AS dot_pct,
    round(sum(CASE WHEN d.runs_batter IN (4, 6) AND NOT d.runs_non_boundary THEN 1 ELSE 0 END) * 100.0
          / nullif(sum(CASE WHEN d.extras_wides = 0 THEN 1 ELSE 0 END), 0), 1)          AS boundary_pct_conceded
FROM main_marts.fct_deliveries d
JOIN main_marts.fct_match_results m ON d.match_id = m.match_id
WHERE d.bowling_team IN ({IN_LIST})
  AND m.event_name ILIKE '%indian premier league%'
  AND m.season = '{SEASON}'
  AND d.innings_number IN (1, 2)
  AND d.over_number BETWEEN 0 AND 19
GROUP BY phase
ORDER BY phase
""")

## Win / Loss Analysis

In [ ]:
# Batting and bowling phase averages split by match result
q(f"""
WITH base AS (
    SELECT
        m.outcome_winner IN ({IN_LIST})  AS won,
        d.over_number,
        d.runs_total,
        d.wicket_count,
        d.extras_wides,
        d.extras_noballs,
        d.runs_batter,
        d.runs_non_boundary,
        d.match_id,
        d.batting_team,
        d.bowling_team
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m ON d.match_id = m.match_id
    WHERE m.event_name ILIKE '%indian premier league%'
      AND m.season = '{SEASON}'
      AND d.innings_number IN (1, 2)
      AND d.over_number BETWEEN 0 AND 19
      AND m.has_result
)
SELECT
    CASE WHEN won THEN 'Won' ELSE 'Lost' END                                             AS result,
    -- batting totals by phase
    round(sum(CASE WHEN batting_team IN ({IN_LIST})
              THEN runs_total ELSE 0 END)::double
          / nullif(count(DISTINCT CASE WHEN batting_team IN ({IN_LIST}) THEN match_id END), 0), 1)
                                                                                         AS bat_avg_score,
    round(sum(CASE WHEN batting_team IN ({IN_LIST}) AND over_number BETWEEN 0  AND 5
              THEN runs_total ELSE 0 END)::double
          / nullif(count(DISTINCT CASE WHEN batting_team IN ({IN_LIST}) THEN match_id END), 0), 1)
                                                                                         AS bat_pp_avg,
    round(sum(CASE WHEN batting_team IN ({IN_LIST}) AND over_number BETWEEN 6  AND 14
              THEN runs_total ELSE 0 END)::double
          / nullif(count(DISTINCT CASE WHEN batting_team IN ({IN_LIST}) THEN match_id END), 0), 1)
                                                                                         AS bat_mid_avg,
    round(sum(CASE WHEN batting_team IN ({IN_LIST}) AND over_number BETWEEN 15 AND 19
              THEN runs_total ELSE 0 END)::double
          / nullif(count(DISTINCT CASE WHEN batting_team IN ({IN_LIST}) THEN match_id END), 0), 1)
                                                                                         AS bat_death_avg,
    -- bowling totals by phase
    round(sum(CASE WHEN bowling_team IN ({IN_LIST})
              THEN runs_total ELSE 0 END)::double
          / nullif(count(DISTINCT CASE WHEN bowling_team IN ({IN_LIST}) THEN match_id END), 0), 1)
                                                                                         AS bowl_avg_conceded,
    round(sum(CASE WHEN bowling_team IN ({IN_LIST}) AND over_number BETWEEN 0  AND 5
              THEN runs_total ELSE 0 END)::double
          / nullif(count(DISTINCT CASE WHEN bowling_team IN ({IN_LIST}) THEN match_id END), 0), 1)
                                                                                         AS bowl_pp_avg,
    round(sum(CASE WHEN bowling_team IN ({IN_LIST}) AND over_number BETWEEN 6  AND 14
              THEN runs_total ELSE 0 END)::double
          / nullif(count(DISTINCT CASE WHEN bowling_team IN ({IN_LIST}) THEN match_id END), 0), 1)
                                                                                         AS bowl_mid_avg,
    round(sum(CASE WHEN bowling_team IN ({IN_LIST}) AND over_number BETWEEN 15 AND 19
              THEN runs_total ELSE 0 END)::double
          / nullif(count(DISTINCT CASE WHEN bowling_team IN ({IN_LIST}) THEN match_id END), 0), 1)
                                                                                         AS bowl_death_avg
FROM base
GROUP BY won
ORDER BY won DESC
""")

## Player Deep-Dive

In [49]:
# ── Set player name here for match-by-match breakdown ────────────────────────
PLAYER = "Abhishek Sharma"
# ─────────────────────────────────────────────────────────────────────────────
print(f"Deep-dive: {PLAYER}  |  {TEAM}  |  {SEASON}")

Deep-dive: Abhishek Sharma  |  Sunrisers Hyderabad  |  2026


In [50]:
# Match-by-match batting for selected player this season
q(f"""
SELECT
    b.match_date,
    CASE WHEN m.team_1 IN ({IN_LIST}) THEN m.team_2 ELSE m.team_1 END  AS vs,
    CASE WHEN m.outcome_winner IN ({IN_LIST}) THEN 'W'
         WHEN m.is_no_result THEN 'NR'
         WHEN m.is_tie       THEN 'T'
         ELSE 'L' END                                                   AS result,
    b.runs,
    b.not_out,
    b.balls_faced                                                        AS balls,
    round(b.runs * 100.0 / nullif(b.balls_faced, 0), 1)                 AS sr,
    b.fours  AS "4s",
    b.sixes  AS "6s"
FROM main_intermediate.int_batting_by_innings b
JOIN main_marts.fct_match_results m ON b.match_id = m.match_id
WHERE b.player_name = '{PLAYER}'
  AND b.batting_team IN ({IN_LIST})
  AND m.event_name ILIKE '%indian premier league%'
  AND m.season = '{SEASON}'
  AND b.innings_number IN (1, 2)
  AND NOT b.super_over
ORDER BY b.match_date
""")

,match_date,vs,result,runs,not_out,balls,sr,4s,6s
0,2026-03-28,Royal Challengers Bengaluru,L,7.0,False,11,63.6,0.0,1.0
1,2026-04-02,Kolkata Knight Riders,W,48.0,False,22,218.2,4.0,4.0
2,2026-04-05,Lucknow Super Giants,L,0.0,False,2,0.0,0.0,0.0
3,2026-04-11,Punjab Kings,L,74.0,False,34,217.6,5.0,8.0
4,2026-04-13,Rajasthan Royals,W,0.0,False,1,0.0,0.0,0.0
5,2026-04-18,Chennai Super Kings,W,59.0,False,23,256.5,6.0,4.0
6,2026-04-21,Delhi Capitals,W,135.0,True,74,182.4,10.0,10.0
7,2026-04-25,Rajasthan Royals,W,57.0,False,33,172.7,11.0,1.0
8,2026-04-29,Mumbai Indians,W,45.0,False,29,155.2,4.0,3.0


In [51]:
# Season batting summary for selected player
q(f"""
SELECT
    count(DISTINCT b.match_id)                                                  AS mat,
    count(*)                                                                    AS inns,
    sum(b.not_out::integer)                                                     AS no,
    sum(b.runs)                                                                 AS runs,
    max(b.runs)                                                                 AS hs,
    round(sum(b.runs)::double / nullif(sum(b.dismissals), 0), 2)                AS avg,
    sum(b.balls_faced)                                                          AS bf,
    round(sum(b.runs) * 100.0 / nullif(sum(b.balls_faced), 0), 2)               AS sr,
    sum(CASE WHEN b.runs >= 50 AND b.runs < 100 THEN 1 ELSE 0 END)              AS "50s",
    sum(CASE WHEN b.runs >= 100 THEN 1 ELSE 0 END)                              AS "100s",
    sum(b.fours)                                                                AS "4s",
    sum(b.sixes)                                                                AS "6s"
FROM main_intermediate.int_batting_by_innings b
JOIN main_marts.fct_match_results m ON b.match_id = m.match_id
WHERE b.player_name = '{PLAYER}'
  AND b.batting_team IN ({IN_LIST})
  AND m.event_name ILIKE '%indian premier league%'
  AND m.season = '{SEASON}'
  AND b.innings_number IN (1, 2)
  AND NOT b.super_over
""")

,mat,inns,no,runs,hs,avg,bf,sr,50s,100s,4s,6s
0,9,9,1.0,425.0,135.0,53.13,229.0,185.59,3.0,1.0,40.0,31.0


In [52]:
# Match-by-match bowling for selected player this season
q(f"""
SELECT
    b.match_date,
    CASE WHEN m.team_1 IN ({IN_LIST}) THEN m.team_2 ELSE m.team_1 END       AS vs,
    CASE WHEN m.outcome_winner IN ({IN_LIST}) THEN 'W'
         WHEN m.is_no_result THEN 'NR'
         WHEN m.is_tie       THEN 'T'
         ELSE 'L' END                                                        AS result,
    (floor(b.legal_deliveries / 6)::integer
     || '.' || (b.legal_deliveries % 6))::varchar                            AS overs,
    b.legal_deliveries                                                        AS balls,
    b.runs_conceded                                                           AS runs,
    b.wickets_credited                                                        AS wkts,
    round(b.runs_conceded * 6.0 / nullif(b.legal_deliveries, 0), 2)          AS econ
FROM main_intermediate.int_bowling_by_innings b
JOIN main_marts.fct_match_results m ON b.match_id = m.match_id
WHERE b.player_name = '{PLAYER}'
  AND b.bowling_team IN ({IN_LIST})
  AND m.event_name ILIKE '%indian premier league%'
  AND m.season = '{SEASON}'
  AND b.innings_number IN (1, 2)
  AND NOT b.super_over
ORDER BY b.match_date
""")

,match_date,vs,result,overs,balls,runs,wkts,econ
0,2026-04-02,Kolkata Knight Riders,W,1.0,6.0,15.0,0.0,15.0
1,2026-04-11,Punjab Kings,L,0.5,5.0,7.0,0.0,8.4
2,2026-04-18,Chennai Super Kings,W,1.0,6.0,13.0,0.0,13.0
